In [ ]:
#-- Packages --#

#--- Operational ---#
import os
import sys 
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import json
import re
import geopandas as gpd

#--- Visualisations ---#
import plotly.express as px
import plotly.graph_objects as go

#-- Directories --#
nb_dir = Path.cwd()
REPO_ROOT = nb_dir.parent
data_dir = REPO_ROOT / 'data/'
docs_dir = REPO_ROOT / 'docs/'
processed_dir = data_dir / 'processed' / 'analysis/'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
#-- Helper Functions --#
def extract_max_year(path):
    """
    Identify shapefile with latest year of available data
    """
    years = re.findall(r"\d{4}", path.stem)
    return max(map(int, years)) if years else -1

In [ ]:
#-- Files --#

# National Canada polygons (GeoJSON)
print(f'Loading National fires GeoJSON...')
fires_GeoJSON_path = processed_dir / "national_canadian_fires/Canada_fires_1990_2024.geojson"
fires_GeoJSON = gpd.read_file(fires_GeoJSON_path)
print(f" National Canada fires GeoJSON loaded. {fires_GeoJSON.crs}\n")

# Canada fires 
fires_dir = processed_dir / "national_canadian_fires"
shp_files = list(fires_dir.glob("*.shp"))

if not shp_files:
    raise FileNotFoundError(f"No shapefiles found in {fires_dir}\n")

fires_path = max(shp_files, key=extract_max_year)

print(f"Loading National fires shapefile... \n File name: {fires_path.name}")
fire_stats = gpd.read_file(fires_path)
print(f" National Canada Fires loaded. {fire_stats.crs}\n")


In [ ]:
fires_GeoJSON.head(1)

In [ ]:
fire_stats.head(5)

In [ ]:
fire_stats.shape


In [ ]:
year_totals = (
    fire_stats.groupby("year")
    .agg(
        fire_count=("fireid", "nunique"),
        total_adj_ha=("adj_ha", "sum"),
    )
)

# De-duplicate to one row per fire per year (keeps its cause)
fires_unique = fire_stats.drop_duplicates(["year", "fireid"])

cause_counts = (
    fires_unique.groupby(["year", "cause"])
    .size()
    .unstack(fill_value=0)
)

fire_years = year_totals.join(cause_counts)

# Percentages using total unique fires as denominator
cause_pct = cause_counts.div(fire_years["fire_count"], axis=0).add_suffix("_pct")*100
fire_years = fire_years.join(cause_pct)

fire_years.head()


In [ ]:
fire_means = fire_years[["fire_count", "total_adj_ha","Human_pct","Natural_pct","Undetermined_pct"]].mean()
fire_means

In [ ]:
fire_means.astype(int)

In [ ]:
fire_means.astype('float').round(2)